# Dataset ranking plots

This notebook reads the saved experiment results from `results/` and regenerates the QQ and comparison plots.

In [ ]:
from pathlib import Path
import pickle

import pandas as pd

try:
    from experiment import prepare_acs, prepare_nhanes
    from plotting import plot_comparison, plot_qq_matrix, plot_target_qq
except ImportError:
    from dataset_ranking.experiment import prepare_acs, prepare_nhanes
    from dataset_ranking.plotting import plot_comparison, plot_qq_matrix, plot_target_qq

BASE_DIR = Path("dataset_ranking") if Path("dataset_ranking").exists() else Path(".")
DATA_DIR = BASE_DIR / "data"
RESULT_DIR = BASE_DIR / "results"
FIGURE_DIR = BASE_DIR / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

In [ ]:
def load_results(name):
    with open(RESULT_DIR / f"{name}_results.pkl", "rb") as file:
        saved = pickle.load(file)
    return saved, saved["summary"]

acs_saved, acs_summary = load_results("acs_income")
nhanes_saved, nhanes_summary = load_results("nhanes_cholesterol")

display(acs_summary)
display(nhanes_summary)

## QQ plots

In [ ]:
acs_data = prepare_acs(DATA_DIR)
nhanes_data = prepare_nhanes(DATA_DIR)

In [ ]:
plot_target_qq(
    acs_data["qq_data"], acs_data["target_id"], acs_data["qq_source_ids"],
    acs_data["site_column"], acs_data["response"], labels=acs_data["qq_labels"],
    target_label="CA", title="Normal QQ Plots: California vs Other States",
    path=FIGURE_DIR / "qqplot_california_vs_other_states.png",
);

plot_qq_matrix(
    acs_data["qq_data"], acs_data["qq_source_ids"], acs_data["site_column"],
    acs_data["response"], labels=acs_data["qq_labels"],
    title="Normal QQ Plots: ACS Income",
    path=FIGURE_DIR / "qqplot_matrix_acs_income.png",
);

In [ ]:
plot_target_qq(
    nhanes_data["qq_data"], nhanes_data["target_id"], nhanes_data["qq_source_ids"],
    nhanes_data["site_column"], nhanes_data["response"],
    labels=nhanes_data["qq_labels"], target_label="Target",
    title="Normal QQ Plots: Target vs Source Datasets (NHANES Cholesterol)",
    path=FIGURE_DIR / "qqplot_target_vs_sources_nhanes.png",
);

## Result plots

In [ ]:
acs_highlights = {
    "duc": {"NY", "NJ"},
    "neg_kl": {"NY"},
    "neg_score_x": {"NY", "NJ", "TX"},
}

plot_comparison(
    acs_summary,
    "Comparison of methods for data selection: ACS Income",
    y_column="avg_rank",
    y_label="Average Rank by Test MSE (1 = Best)",
    highlight_map=acs_highlights,
    path=FIGURE_DIR / "fig3.png",
);

In [ ]:
plot_comparison(
    nhanes_summary,
    "Comparison of methods for data selection: NHANES Cholesterol",
    y_column="avg_rank",
    y_label="Average Rank of Test MSE (1 = Best)",
    path=FIGURE_DIR / "fig_nhanes.png",
);

plot_comparison(
    nhanes_summary,
    "Comparison of methods for data selection: NHANES Cholesterol",
    y_column="weighted_mse",
    y_label="Avg Test MSE",
    path=FIGURE_DIR / "fig_nhanes_mse.png",
);